In [ ]:
using Bessels, MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.5127532
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
t_max = 65/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=1/(sqrt(2)*10*ω_a)
t_save = max(t_min_save,0):dt_save:t_max

re_a_0, im_a_0 = 9, -4
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
re_b_0, im_b_0 = 0, 0
ϕ_b_0, q_b_0=sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
system_prepared = prepare(system)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)
environment = MeanField(κ_a, κ_b)
environment_prepared = prepare(system,environment);

u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

ϕ_a, ϕ_b, q_a, q_b, t_list = default_solver((u_dot_0, u_0), (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

In [ ]:
p1,p2 = plot_mode(ϕ_a, q_a, t_list, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
p1,p2 = plot_mode(ϕ_b, q_b, t_list, ω_b_0, Z_b)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
Re = ϕ_b / sqrt(2*Z_b)
Im = q_b * sqrt(Z_b/2);

In [ ]:
ω =  ω_b_0
Re_2 =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
Im_2 =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list);

In [ ]:
start, step, ending = 1, 1, 100000
center = (sum(Re_2)/length(Re_2),sum(Im_2)/length(Im_2))
scatter(Re_2[start:step:ending].-center[1],Im_2[start:step:ending].-center[2], zcolor = t_list[start:step:ending].*κ_b, ms=1, cmap = Perso_colormap, label="")
plot!(xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",colorbar_title="\n"*L"\kappa_bt")
#savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/chaos/lobe_omega_b.pdf")

In [ ]:
t_max = 65/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max

re_a_0, im_a_0 = 9, -4
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
re_b_0, im_b_0 = 0, 0
ϕ_b_0, q_b_0=sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
system_prepared = prepare(system)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)
environment = MeanField(κ_a, κ_b)
environment_prepared = prepare(system,environment);

u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

ϕ_a, ϕ_b, q_a, q_b, t_list = default_solver((u_dot_0, u_0), (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

In [ ]:
Re = ϕ_b / sqrt(2*Z_b)
Im = q_b * sqrt(Z_b/2);
ω =  ω_b_0
Re_2 =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
Im_2 =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list);
start, step, ending = 1, 1, length(Re_2)
center = (sum(Re_2)/length(Re_2),sum(Im_2)/length(Im_2))
scatter(Re_2[start:step:ending].-center[1],Im_2[start:step:ending].-center[2], zcolor = t_list[start:step:ending].*κ_b, ms=1, cmap = Perso_colormap, label="")
plot!(xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",colorbar_title="\n"*L"\kappa_bt")

In [ ]:
t_max = 65/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_b_0)
t_save = max(t_min_save,0):dt_save:t_max

re_a_0, im_a_0 = 9, -4
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
re_b_0, im_b_0 = 0, 0
ϕ_b_0, q_b_0=sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
system_prepared = prepare(system)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)
environment = MeanField(κ_a, κ_b)
environment_prepared = prepare(system,environment);

u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

ϕ_a, ϕ_b, q_a, q_b, t_list = default_solver((u_dot_0, u_0), (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

In [ ]:
Re = ϕ_b / sqrt(2*Z_b)
Im = q_b * sqrt(Z_b/2);
ω =  ω_b_0
Re_2 =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
Im_2 =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list);
start, step, ending = 1, 1, length(Re_2)
center = (sum(Re_2)/length(Re_2),sum(Im_2)/length(Im_2))
scatter(Re_2[start:step:ending].-center[1],Im_2[start:step:ending].-center[2], zcolor = t_list[start:step:ending].*κ_b, ms=1, cmap = Perso_colormap, label="")
plot!(xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",colorbar_title="\n"*L"\kappa_bt")

In [ ]:
t_max = 65/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(200ω_p)
t_save = max(t_min_save,0):dt_save:t_max

re_a_0, im_a_0 = 9, -4
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
re_b_0, im_b_0 = 0, 0
ϕ_b_0, q_b_0=sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
system_prepared = prepare(system)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)
environment = MeanField(κ_a, κ_b)
environment_prepared = prepare(system,environment);

u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

ϕ_a, ϕ_b, q_a, q_b, t_list = default_solver((u_dot_0, u_0), (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

In [ ]:
Re = ϕ_b / sqrt(2*Z_b)
Im = q_b * sqrt(Z_b/2);
ω =  ω_b_0
Re_2 =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
Im_2 =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list);
step, ending =50, length(Re_2)
center = (sum(Re_2)/length(Re_2),sum(Im_2)/length(Im_2))

In [ ]:

for start in range(1,50)
    p=scatter(Re_2[start:step:ending].-center[1],Im_2[start:step:ending].-center[2], zcolor = t_list[start:step:ending].*κ_b, ms=1, cmap = Perso_colormap, label="")
    plot!(p,xlabel=L"\Re(a)", ylabel=L"\Im(a)", title=L"g_2/2\pi="*string(round(g2/2pi*1000,digits=3))*" MHz",colorbar_title="\n"*L"\kappa_bt")
    xlims!(p,-2.5, 3)
    ylims!(p,-3, 2.5)
    display(plot(p))
end

In [ ]:
R = 5
θ_list = range(start = 0, stop = 2pi, length = 100)
step, ending = 100, length(Re_2)
center = (sum(Re_2)/length(Re_2), sum(Im_2)/length(Im_2))

x = Float64[]
y = Float64[]
z = Float64[]
t_tot = Float64[]

for start in 1:100
    x_0 = Re_2[start:step:ending] .- center[1]
    z_0 = Im_2[start:step:ending] .- center[2]

    θ = θ_list[start]
    x_rot = (R .+ x_0) .* cos(θ)
    y_rot = (R .+ x_0) .* sin(θ)

    append!(x, x_rot)
    append!(y, y_rot)
    append!(z, z_0)
    append!(t_tot,t_list[start:step:ending])
end

In [ ]:
using CSV, DataFrames

# assume x, y, z, t are vectors of the same length
df = DataFrame(x=x, y=y, z=z, t=t_tot)

CSV.write("../ouput/torus_25_11_2025_b.csv", df)